# Семинар 1. Линейная регрессия

#### Шаг 1. Загрузка датасета и вывод на экран

In [1]:
import pandas as pd

data_frame = pd.read_csv("delivery_dataset.csv")
data_frame.head(5)

#### Шаг2. Разделение выборки на тренировочную, валидационную и тестовую с выбором нужных столбцов:
* в тренировочной: 300 первых заказов;
* в валидационной: 100 следующих; 
* в тестовой: 100 последних.

In [2]:
features = [
    "Расстояние до клиента (в км)",
    "Количество позиций в чеке (в шт.)",
    "Балл пробок на дорогах (в баллах)",
    "Погода (в градусах Цельсия)",
    "Этаж доставки (в этажах)"
]
x = data_frame[features]
y = data_frame["Время доставки (в минутах)"]

# Делим по порядку строк, как в задании.
x_train, y_train = x.iloc[:300], y.iloc[:300]
x_val, y_val = x.iloc[300:400], y.iloc[300:400]
x_test, y_test = x.iloc[400:500], y.iloc[400:500]

#### Вопрос 1. 

Для чего мы разбиваем данные на выборки? Для чего нам нужна тренировачная, для чего валидационная, для чего тестовая?

**Ответ:** train — чтобы модель училась, val — чтобы по ходу проверять, не чудит ли она, а test — финальная проверка. Короче, так честнее видно, как модель работает на новых данных.

#### Шаг 3. Установка sklearn чтобы обучать модель линейной регрессии через Ridge

In [3]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Шаг 4. Обучение модели линейной регрессии на тренировочной выборке с помощью Ridge и sparse_cg

In [4]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0, solver="sparse_cg", max_iter=1000, tol=1e-6)
model.fit(x_train, y_train)

#### Шаг 5. Вывод на экран графиков ошибок (SSE, MSE, RMSE) в зависимости от итераций для train и val выборок.

In [5]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

iterations = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
train_sse, val_sse = [], []
train_mse, val_mse = [], []
train_rmse, val_rmse = [], []

for max_iter in iterations:
    current_model = Ridge(alpha=1.0, solver="sparse_cg", max_iter=max_iter, tol=1e-6)
    current_model.fit(x_train, y_train)

    train_predictions = current_model.predict(x_train)
    val_predictions = current_model.predict(x_val)
    train_mse_value = mean_squared_error(y_train, train_predictions)
    val_mse_value = mean_squared_error(y_val, val_predictions)

    train_sse.append(train_mse_value * len(y_train))
    val_sse.append(val_mse_value * len(y_val))
    train_mse.append(train_mse_value)
    val_mse.append(val_mse_value)
    train_rmse.append(train_mse_value ** 0.5)
    val_rmse.append(val_mse_value ** 0.5)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for axis, train_errors, val_errors, metric_name in zip(
    axes,
    [train_sse, train_mse, train_rmse],
    [val_sse, val_mse, val_rmse],
    ["SSE", "MSE", "RMSE"]
):
    axis.plot(iterations, train_errors, label="train")
    axis.plot(iterations, val_errors, label="val")
    axis.set_title(metric_name)
    axis.set_xlabel("Число итераций")
    axis.set_ylabel("Ошибка")
    axis.legend()
    axis.grid()

plt.show()

#### Вопрос 2.

Какую информацию про нашу модель несут эти графики?

**Ответ:** видно, уменьшается ли ошибка и есть ли разница между train и val. Если val сильно хуже train, то модель, скорее всего, уже запомнила train, короче.

#### Вопрос 3. 

Какая траектория линий train и val ошибок будет на графиках недообученной модели и почему?

Какая траектория линий train и val ошибок будет на графиках обученной модели и почему?

Какая траектория линий train и val ошибок будет на графиках переобученной модели и почему?

**Ответ:** при недообучении обе ошибки высокие — модель ещё не въехала. У нормальной модели они низкие и примерно рядом. При переобучении train падает, а val перестаёт падать или растёт: модель слишком залипла на train.

#### Шаг 6. Применение модели на тестовой выборке

In [6]:
test_predictions = model.predict(x_test)
test_predictions[:5]

#### Шаг 7. RMSE на тестовой выборке

In [7]:
test_rmse = mean_squared_error(y_test, test_predictions) ** 0.5
print(f"RMSE на тестовой выборке: {test_rmse:.2f}")

#### Вопрос 4.

Что получившийся RMSE может сказать о нашей модели?

**Ответ:** RMSE показывает, насколько минут в среднем модель промахивается. Чем меньше, тем лучше; надо ещё сравнить с тем, насколько вообще разбросано время доставки.